# DAM - Prueba de Concepto

- **Extraccion de metadatos EXIF**: lee los datos tecnicos de la
  camara directamente del archivo de imagen (nada de transcripcion manual).
- **Analisis con el modelo (cadena de 3 pasos)**: el modelo de
  vision analiza el contenido, lo clasifica comercialmente y evalua la
  calidad/coherencia del resultado.
- **Revision del editor**: muestra el resultado combinado (EXIF +
  analisis) en un formato legible, listo para que un editor lo apruebe.

**Como usarlo**
1. Corre la celda de instalacion de dependencias (una vez).
2. Corre las celdas en orden hasta la seccion "Configuracion" y ajusta
   `INPUT_DIR` (carpeta con tus fotos y, opcionalmente, un `.txt` de notas).
3. Corre el resto de las celdas: vas a ver el resultado de la Etapa 6 al
   final, y un archivo `analyzed_photos.jsonl` con el resultado completo.


## 0. Dependencias

Se necesita `exifread` (Etapa 2) y `openai` (Etapa 3). `rawpy` y `Pillow`.

In [1]:
!pip install -q exifread openai pillow rawpy


## 1. Imports y logging

In [2]:
from __future__ import annotations

import base64
import getpass
import json
import logging
import mimetypes
import os
import time
import uuid
from dataclasses import asdict, dataclass, field
from fractions import Fraction
from pathlib import Path
from typing import Dict, List, Optional

import exifread
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("dam_poc")


## 2. Configuracion

- `INPUT_DIR`: carpeta con las imagenes a procesar (y, opcionalmente, un
  archivo `.txt` con notas del fotografo — ver formato mas abajo).
- `OUTPUT_JSONL`: donde se guarda el resultado final de la Etapa 3+6.
- `MODEL_ID`: modelo de vision usado en la Etapa 3.
- `CONFIDENCE_REVIEW_THRESHOLD`: por debajo de este puntaje, la foto queda
  marcada para revision humana en la Etapa 6.

La API key de OpenAI se pide de forma interactiva (`getpass`) y no queda
guardada en el notebook.

In [3]:
INPUT_DIR = Path("fotos")          # carpeta con las imagenes del lote
OUTPUT_JSONL = Path("analyzed_photos_poc.jsonl")

MODEL_ID = "gpt-5.6-luna"
MAX_RETRIES = 2
CONFIDENCE_REVIEW_THRESHOLD = 0.80

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp",  # comprimidas / directas al modelo
    ".raw", ".cr2", ".cr3", ".nef", ".arw", ".dng", ".orf", ".rw2",  # raw
}

In [4]:
def _get_api_key() -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
    return os.environ["OPENAI_API_KEY"]


_client: Optional[OpenAI] = None


def get_client() -> OpenAI:
    global _client
    if _client is None:
        _client = OpenAI(api_key=_get_api_key())
    return _client


## 3. Modelos de datos

Mismo esquema que usa el pipeline completo: metadata EXIF, analisis de
contenido, clasificacion comercial y control de calidad.

In [5]:
@dataclass
class ExifMetadata:
    camera_brand: Optional[str] = None
    camera_model: Optional[str] = None
    iso: Optional[int] = None
    aperture: Optional[float] = None
    focal_length_mm: Optional[int] = None

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class StagedPhoto:
    """Foto + su EXIF ya extraido (salida de la Etapa 2)."""

    photo_id: str
    image_path: Path
    photographer_notes: str
    exif_metadata: ExifMetadata
    exif_warnings: list = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "photo_id": self.photo_id,
            "image_path": str(self.image_path),
            "photographer_notes": self.photographer_notes,
            "exif_metadata": self.exif_metadata.to_dict(),
            "exif_warnings": self.exif_warnings,
        }


@dataclass
class ContentAnalysis:
    primary_subject: Optional[str] = None
    keywords: List[str] = field(default_factory=list)
    environment: Optional[str] = None  # indoor | outdoor | studio | unknown
    color_palette: List[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class CommercialClassification:
    primary_category: Optional[str] = None
    secondary_category: Optional[str] = None

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class QualityControl:
    confidence_score: Optional[float] = None
    flagged_for_review: Optional[bool] = None
    review_reason: Optional[str] = None

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class AnalyzedPhoto:
    """StagedPhoto + el analisis del modelo (salida de la Etapa 3, entrada de la Etapa 6)."""

    photo_id: str
    image_path: Path
    photographer_notes: str
    exif_metadata: ExifMetadata
    content_analysis: ContentAnalysis
    commercial_classification: CommercialClassification
    quality_control: QualityControl

    def to_dict(self) -> dict:
        return {
            "photo_id": self.photo_id,
            "image_path": str(self.image_path),
            "photographer_notes": self.photographer_notes,
            "exif_metadata": self.exif_metadata.to_dict(),
            "content_analysis": self.content_analysis.to_dict(),
            "commercial_classification": self.commercial_classification.to_dict(),
            "quality_control": self.quality_control.to_dict(),
        }


## 4. Carga de imagenes (utilidad minima)

Esto **no** es la Etapa 1 completa del pipeline; es solo el minimo
necesario para poder alimentar la Etapa 2 sin depender de un JSONL
generado por otro script. Empareja cada imagen de `INPUT_DIR` con sus
notas si encuentra un `.txt`:

- Formato etiquetado (recomendado): `nombre_archivo.jpg: notas...` por linea.
- Si no matchea, intenta bloques separados por lineas en blanco, en el
  mismo orden que las imagenes (ordenadas alfabeticamente).
- Si nada de eso aplica, usa notas vacias.

In [6]:
def _find_images(input_dir: Path) -> List[Path]:
    if not input_dir.is_dir():
        raise FileNotFoundError(f"La carpeta de entrada no existe: {input_dir}")
    images = sorted(
        p for p in input_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not images:
        raise FileNotFoundError(f"No se encontraron imagenes en: {input_dir}")
    return images


def _find_notes_file(input_dir: Path) -> Optional[Path]:
    txt_files = sorted(input_dir.glob("*.txt"))
    return txt_files[0] if txt_files else None


def _parse_tagged_notes(raw_text: str, image_names: List[str]) -> Optional[Dict[str, str]]:
    lookup = {name.lower(): name for name in image_names}
    notes_by_image: Dict[str, str] = {}
    current_key: Optional[str] = None

    for line in raw_text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        head, sep, rest = stripped.partition(":")
        candidate = head.strip().lower()
        if sep and candidate in lookup:
            current_key = lookup[candidate]
            notes_by_image[current_key] = rest.strip()
        elif current_key is not None:
            notes_by_image[current_key] = (notes_by_image[current_key] + " " + stripped).strip()

    return notes_by_image or None


def _parse_blocks_in_order(raw_text: str, image_names: List[str]) -> Optional[Dict[str, str]]:
    blocks = [b.strip() for b in raw_text.split("\n\n") if b.strip()]
    if len(blocks) != len(image_names):
        return None
    return dict(zip(image_names, blocks))


def _parse_notes(notes_path: Optional[Path], image_names: List[str]) -> Dict[str, str]:
    if notes_path is None:
        logger.warning("No se encontro archivo de notas (.txt); se ingresa con notas vacias.")
        return {name: "" for name in image_names}

    raw_text = notes_path.read_text(encoding="utf-8", errors="replace")

    tagged = _parse_tagged_notes(raw_text, image_names)
    if tagged is not None:
        for name in image_names:
            tagged.setdefault(name, "")
        return tagged

    by_blocks = _parse_blocks_in_order(raw_text, image_names)
    if by_blocks is not None:
        return by_blocks

    logger.warning(
        "No se pudo emparejar el archivo de notas por imagen; se asigna el "
        "texto completo a todas las fotos del lote."
    )
    return {name: raw_text.strip() for name in image_names}


def load_images(input_dir: str | Path) -> List[dict]:
    """Devuelve una lista de {photo_id, image_path, photographer_notes}."""
    input_dir = Path(input_dir)
    images = _find_images(input_dir)
    notes_by_name = _parse_notes(_find_notes_file(input_dir), [img.name for img in images])

    return [
        {
            "photo_id": uuid.uuid4().hex[:10],
            "image_path": img,
            "photographer_notes": notes_by_name.get(img.name, ""),
        }
        for img in images
    ]


## 5. Etapa 2 - Extraccion de metadatos EXIF

Lee el EXIF directamente del archivo (sin intervencion del modelo). Los
tags ausentes quedan en `None` con una advertencia, nunca se inventan
datos.

In [7]:
def _tag_str(tags: dict, key: str) -> Optional[str]:
    value = tags.get(key)
    return str(value).strip() if value is not None else None


def _tag_int(tags: dict, key: str) -> Optional[int]:
    value = tags.get(key)
    if value is None:
        return None
    try:
        return int(Fraction(str(value)))
    except (ValueError, ZeroDivisionError):
        return None


def _tag_float(tags: dict, key: str) -> Optional[float]:
    value = tags.get(key)
    if value is None:
        return None
    try:
        return round(float(Fraction(str(value))), 2)
    except (ValueError, ZeroDivisionError):
        return None


def extract_exif(image_path: str | Path) -> tuple[ExifMetadata, List[str]]:
    image_path = Path(image_path)
    warnings: List[str] = []

    with open(image_path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    if not tags:
        warnings.append("El archivo no tiene datos EXIF legibles.")
        return ExifMetadata(), warnings

    metadata = ExifMetadata(
        camera_brand=_tag_str(tags, "Image Make"),
        camera_model=_tag_str(tags, "Image Model"),
        iso=_tag_int(tags, "EXIF ISOSpeedRatings"),
        aperture=_tag_float(tags, "EXIF FNumber"),
        focal_length_mm=_tag_int(tags, "EXIF FocalLength"),
    )

    for field_name, value in metadata.to_dict().items():
        if value is None:
            warnings.append(f"Tag EXIF faltante para '{field_name}'.")

    return metadata, warnings


def enrich_with_exif(photos: List[dict]) -> List[StagedPhoto]:
    """Ejecuta la Etapa 2 sobre la lista de imagenes cargadas."""
    staged: List[StagedPhoto] = []

    for photo in photos:
        try:
            metadata, warnings = extract_exif(photo["image_path"])
        except Exception as exc:  # archivo corrupto, formato no soportado, etc.
            logger.warning("No se pudo leer EXIF de %s: %s", photo["image_path"], exc)
            metadata, warnings = ExifMetadata(), [f"Error al leer EXIF: {exc}"]

        staged.append(
            StagedPhoto(
                photo_id=photo["photo_id"],
                image_path=photo["image_path"],
                photographer_notes=photo["photographer_notes"],
                exif_metadata=metadata,
                exif_warnings=warnings,
            )
        )

    logger.info("Etapa 2 completada: EXIF procesado para %d foto(s).", len(staged))
    return staged


## 6. Etapas 3 y 4 - Analisis con el modelo (cadena de 3 pasos)

Workflow fijo (no es un agente): 3 llamadas secuenciales, cada una recibe
el resultado de la anterior como contexto.

1. `analyze_content` -> `content_analysis`
2. `classify_category` (usa el paso 1) -> `commercial_classification`
3. `assess_quality` (usa los pasos 1 y 2) -> `quality_control`

In [8]:
class LLMCallError(RuntimeError):
    """El modelo no devolvio una respuesta usable tras los reintentos."""


_DIRECT_IMAGE_TYPES = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
    ".webp": "image/webp",
}
_RAW_IMAGE_TYPES = {".raw", ".cr2", ".cr3", ".nef", ".arw", ".dng", ".orf", ".rw2"}


def _raw_to_jpeg_bytes(path: Path) -> bytes:
    try:
        import rawpy  # type: ignore
    except ImportError as exc:
        raise LLMCallError(
            f"'{path.name}' es un archivo RAW y se necesita el paquete "
            "'rawpy' (pip install rawpy) para revelarlo a JPEG antes de "
            "mandarlo al modelo."
        ) from exc

    import io
    from PIL import Image

    with rawpy.imread(str(path)) as raw:
        rgb = raw.postprocess()

    buffer = io.BytesIO()
    Image.fromarray(rgb).save(buffer, format="JPEG", quality=90)
    return buffer.getvalue()


def _image_to_data_url(image_path: Path) -> str:
    suffix = image_path.suffix.lower()

    if suffix in _DIRECT_IMAGE_TYPES:
        mime = _DIRECT_IMAGE_TYPES[suffix]
        raw_bytes = image_path.read_bytes()
    elif suffix in _RAW_IMAGE_TYPES:
        mime = "image/jpeg"
        raw_bytes = _raw_to_jpeg_bytes(image_path)
    else:
        mime = mimetypes.guess_type(str(image_path))[0] or "image/jpeg"
        raw_bytes = image_path.read_bytes()
        logger.warning("Tipo de imagen no reconocido para %s; se envia como %s.", image_path, mime)

    encoded = base64.b64encode(raw_bytes).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def call_json(system_prompt: str, user_text: str, image_path: Optional[Path] = None) -> dict:
    """Llama al modelo pidiendo JSON y lo parsea, con reintentos."""
    client = get_client()

    content: list = [{"type": "text", "text": user_text}]
    if image_path is not None:
        content.append({
            "type": "image_url",
            "image_url": {"url": _image_to_data_url(image_path)},
        })

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": content},
    ]

    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content
        try:
            return json.loads(raw)
        except (json.JSONDecodeError, TypeError) as exc:
            last_error = exc
            logger.warning("Intento %d/%d: el modelo no devolvio JSON valido (%s).", attempt, MAX_RETRIES, exc)

    raise LLMCallError(f"El modelo no devolvio JSON valido tras {MAX_RETRIES} intentos: {last_error}")


In [9]:
_CONTENT_SYSTEM_PROMPT = """
Eres un analista visual para un DAM (Digital Asset Management) de fotografia.
Tu tarea es analizar UNA fotografia junto a las notas sueltas del fotografo
y describir su contenido. No inventes detalles que no se vean en la imagen
ni esten respaldados por las notas.

Devuelve UNICAMENTE un objeto JSON con este esquema, sin texto adicional:
{
  "primary_subject": "string, breve y concreto",
  "keywords": ["10 a 15 terminos descriptivos en espanol: objetos, ambiente, estilo, uso comercial. Sin 'foto' ni 'imagen'."],
  "environment": "uno de: indoor | outdoor | studio | unknown",
  "color_palette": ["3 a 5 colores dominantes, como codigo hex o nombre estandar"]
}
""".strip()

_CLASSIFICATION_SYSTEM_PROMPT = """
Eres un clasificador comercial de fotografia para un DAM. Recibes la imagen
y el analisis de contenido ya extraido (sujeto, keywords, ambiente). Con
eso, asigna la categoria comercial mas adecuada.

Categorias sugeridas (usa la que mejor calce; puedes usar otra si ninguna
aplica bien): Food & Beverage, Lifestyle, Corporate, Nature, Technology,
Architecture, Fashion, Travel.

Devuelve UNICAMENTE un objeto JSON con este esquema, sin texto adicional:
{
  "primary_category": "string",
  "secondary_category": "string, o null si no aplica una subcategoria clara"
}
""".strip()

_QUALITY_SYSTEM_PROMPT = """
Eres un revisor de control de calidad para un DAM. Recibes las notas
originales del fotografo, la imagen, y el analisis + clasificacion que ya
se generaron para esta foto. Evalua que tan coherente es ese analisis con
la imagen real y con las notas.

Devuelve UNICAMENTE un objeto JSON con este esquema, sin texto adicional:
{
  "confidence_score": "float entre 0.00 y 1.00, segun la coherencia entre notas, imagen y el analisis generado",
  "review_reason": "string breve explicando dudas o contradicciones, o null si no hay ninguna"
}

No decidas tu mismo si hay que marcar la foto para revision: eso lo
calcula el sistema a partir del confidence_score que devuelvas.
""".strip()


def analyze_content(photo: StagedPhoto) -> ContentAnalysis:
    user_text = (
        "Notas del fotografo para esta imagen:\n"
        f"{photo.photographer_notes or '(sin notas)'}"
    )
    data = call_json(_CONTENT_SYSTEM_PROMPT, user_text, image_path=photo.image_path)
    return ContentAnalysis(
        primary_subject=data.get("primary_subject"),
        keywords=list(data.get("keywords") or []),
        environment=data.get("environment"),
        color_palette=list(data.get("color_palette") or []),
    )


def classify_category(photo: StagedPhoto, content: ContentAnalysis) -> CommercialClassification:
    user_text = (
        "Analisis de contenido ya extraido para esta imagen:\n"
        f"- primary_subject: {content.primary_subject}\n"
        f"- keywords: {', '.join(content.keywords)}\n"
        f"- environment: {content.environment}\n"
    )
    data = call_json(_CLASSIFICATION_SYSTEM_PROMPT, user_text, image_path=photo.image_path)
    return CommercialClassification(
        primary_category=data.get("primary_category"),
        secondary_category=data.get("secondary_category"),
    )


def assess_quality(
    photo: StagedPhoto,
    content: ContentAnalysis,
    classification: CommercialClassification,
) -> QualityControl:
    user_text = (
        "Notas originales del fotografo:\n"
        f"{photo.photographer_notes or '(sin notas)'}\n\n"
        "Analisis generado para esta imagen:\n"
        f"- primary_subject: {content.primary_subject}\n"
        f"- keywords: {', '.join(content.keywords)}\n"
        f"- environment: {content.environment}\n"
        f"- color_palette: {', '.join(content.color_palette)}\n"
        f"- primary_category: {classification.primary_category}\n"
        f"- secondary_category: {classification.secondary_category}\n"
    )
    data = call_json(_QUALITY_SYSTEM_PROMPT, user_text, image_path=photo.image_path)

    confidence = data.get("confidence_score")
    try:
        confidence = float(confidence) if confidence is not None else None
    except (TypeError, ValueError):
        confidence = None

    flagged = confidence is None or confidence < CONFIDENCE_REVIEW_THRESHOLD

    return QualityControl(
        confidence_score=confidence,
        flagged_for_review=flagged,
        review_reason=data.get("review_reason"),
    )


def analyze_photo(photo: StagedPhoto) -> AnalyzedPhoto:
    """Corre la cadena completa (3 llamadas) para una sola foto."""
    content = analyze_content(photo)
    classification = classify_category(photo, content)
    quality = assess_quality(photo, content, classification)

    return AnalyzedPhoto(
        photo_id=photo.photo_id,
        image_path=photo.image_path,
        photographer_notes=photo.photographer_notes,
        exif_metadata=photo.exif_metadata,
        content_analysis=content,
        commercial_classification=classification,
        quality_control=quality,
    )


def analyze_batch(photos: List[StagedPhoto]) -> List[AnalyzedPhoto]:
    """Etapa 3 para todo el lote. Una foto que falle no aborta el resto."""
    results: List[AnalyzedPhoto] = []
    for photo in photos:
        try:
            results.append(analyze_photo(photo))
        except LLMCallError as exc:
            logger.error("Etapa 3 fallo para %s: %s", photo.image_path, exc)

    logger.info("Etapa 3 completada: %d/%d foto(s) analizada(s).", len(results), len(photos))
    return results


## 7. Etapa 6 - Revision del editor

Muestra el resultado combinado (EXIF + analisis del modelo) en un formato
legible, a modo de placeholder de la pantalla de aprobacion del editor.

In [10]:
_WIDTH = 70


def format_photo_result(photo: AnalyzedPhoto) -> str:
    exif = photo.exif_metadata
    content = photo.content_analysis
    classification = photo.commercial_classification
    quality = photo.quality_control

    camera = " ".join(p for p in (exif.camera_brand, exif.camera_model) if p) or "-"
    aperture = f"f/{exif.aperture}" if exif.aperture is not None else "-"
    focal = f"{exif.focal_length_mm}mm" if exif.focal_length_mm is not None else "-"
    iso = exif.iso if exif.iso is not None else "-"

    keywords = ", ".join(content.keywords) if content.keywords else "-"
    colors = ", ".join(content.color_palette) if content.color_palette else "-"

    revisar = "SI" if quality.flagged_for_review else "No"
    confianza = f"{quality.confidence_score:.2f}" if quality.confidence_score is not None else "-"
    motivo = quality.review_reason or "-"

    lines = [
        "=" * _WIDTH,
        f"Foto: {photo.photo_id}  ({Path(photo.image_path).name})",
        "=" * _WIDTH,
        "Notas del fotografo:",
        f"  {photo.photographer_notes or '(sin notas)'}",
        "",
        "Metadatos EXIF",
        f"  Camara       : {camera}",
        f"  ISO          : {iso}",
        f"  Apertura     : {aperture}",
        f"  Focal        : {focal}",
        "",
        "Analisis de contenido",
        f"  Sujeto       : {content.primary_subject or '-'}",
        f"  Ambiente     : {content.environment or '-'}",
        f"  Keywords     : {keywords}",
        f"  Colores      : {colors}",
        "",
        "Clasificacion comercial",
        f"  Categoria    : {classification.primary_category or '-'}",
        f"  Subcategoria : {classification.secondary_category or '-'}",
        "",
        "Control de calidad",
        f"  Confianza    : {confianza}",
        f"  Revisar      : {revisar}",
        f"  Motivo       : {motivo}",
        "=" * _WIDTH,
    ]
    return "\n".join(lines)


def print_photo_result(photo: AnalyzedPhoto) -> None:
    print(format_photo_result(photo))


def print_batch_results(photos: List[AnalyzedPhoto]) -> None:
    """Etapa 6: muestra todo el lote analizado para revision del editor."""
    if not photos:
        print("No hay fotos para mostrar.")
        return

    for photo in photos:
        print_photo_result(photo)
        print()

    flagged = sum(1 for p in photos if p.quality_control.flagged_for_review)
    print(f"Total: {len(photos)} foto(s) - {flagged} marcada(s) para revision.")


## 8. Ejecucion end-to-end

Corre Etapa 2 -> Etapa 3 -> Etapa 4 -> Etapa 6 sobre `INPUT_DIR`. La primera vez va a
pedir la `OPENAI_API_KEY` de forma interactiva.

Se mide el tiempo total del lote (`elapsed_seconds`) para poder compararlo
despues contra la meta de negocio de la Ficha de Caso de Uso ("tiempo de
procesamiento y catalogacion por lote menor a 5 minutos").

In [11]:
batch_start = time.time()

raw_images = load_images(INPUT_DIR)
staged_photos = enrich_with_exif(raw_images)
analyzed_photos = analyze_batch(staged_photos)

elapsed_seconds = time.time() - batch_start

print_batch_results(analyzed_photos)

print()
print(
    f"Tiempo total del lote: {elapsed_seconds:.1f} segundos "
    f"({elapsed_seconds/60:.2f} minutos) para {len(analyzed_photos)} foto(s) "
    f"({elapsed_seconds/max(len(analyzed_photos), 1):.1f} s/foto)."
)


OPENAI_API_KEY: ··········
Foto: 9b71738140  (Foto1.JPEG)
Notas del fotografo:
  Monumento a Urquiza en Palermo, CABA, Argentina

Metadatos EXIF
  Camara       : Apple iPhone 16
  ISO          : 160
  Apertura     : f/1.6
  Focal        : 5mm

Analisis de contenido
  Sujeto       : Monumento a Urquiza en Palermo
  Ambiente     : outdoor
  Keywords     : Monumento a Urquiza, escultura monumental, General Urquiza, Palermo, Buenos Aires, CABA, Argentina, atardecer, cielo nublado, edificios urbanos, paisaje urbano, avenida, tránsito vehicular, motocicleta, espacio público
  Colores      : dorado, blanco, gris, verde oscuro, azul grisáceo

Clasificacion comercial
  Categoria    : Travel
  Subcategoria : Architecture

Control de calidad
  Confianza    : 0.99
  Revisar      : No
  Motivo       : -

Foto: 19e724e206  (Foto2.jpeg)
Notas del fotografo:
  Perro blanco llamado Pecas

Metadatos EXIF
  Camara       : -
  ISO          : -
  Apertura     : -
  Focal        : -

Analisis de contenido
 

## 9. Resultado de negocio (KPI vs. Pasaporte del PoC)

Traduce la evidencia tecnica de la celda anterior a las metas de negocio
declaradas en el Pasaporte del PoC / Ficha de Caso de Uso (Sesion 1, Paso 2
"Resultado deseado"). Esto es evidencia de negocio, separada de la
evidencia tecnica de ejecucion de arriba.

In [12]:
KPI_MAX_SECONDS_PER_BATCH = 5 * 60  # meta: tiempo por lote < 5 minutos
KPI_MIN_KEYWORDS = 10                # meta: 10 a 15 keywords relevantes por foto

cumple_tiempo = elapsed_seconds <= KPI_MAX_SECONDS_PER_BATCH
fotos_con_keywords_suficientes = sum(
    1 for p in analyzed_photos if len(p.content_analysis.keywords) >= KPI_MIN_KEYWORDS
)

print("=" * _WIDTH)
print("KPI 1) Tiempo de procesamiento por lote < 5 minutos")
print(
    f"       {'CUMPLE' if cumple_tiempo else 'NO CUMPLE'} -> {elapsed_seconds:.1f}s "
    f"de {KPI_MAX_SECONDS_PER_BATCH}s permitidos, para {len(analyzed_photos)} foto(s)."
)
print()
print(f"KPI 2) Al menos {KPI_MIN_KEYWORDS} keywords relevantes por foto")
print(f"       {fotos_con_keywords_suficientes}/{len(analyzed_photos)} foto(s) cumplen.")
print()
print("KPI 3) 0% errores de transcripcion manual de metadatos EXIF")
print("       Estructural: el dato se lee del archivo por script, nunca se transcribe a mano.")
print("=" * _WIDTH)


KPI 1) Tiempo de procesamiento por lote < 5 minutos
       CUMPLE -> 189.3s de 300s permitidos, para 5 foto(s).

KPI 2) Al menos 10 keywords relevantes por foto
       5/5 foto(s) cumplen.

KPI 3) 0% errores de transcripcion manual de metadatos EXIF
       Estructural: el dato se lee del archivo por script, nunca se transcribe a mano.


## 10. Guardar resultados

Guarda el lote analizado como JSON Lines, una linea por foto.

In [13]:
with OUTPUT_JSONL.open("w", encoding="utf-8") as f:
    for photo in analyzed_photos:
        f.write(json.dumps(photo.to_dict(), ensure_ascii=False) + "\n")

print(f"Listo. {OUTPUT_JSONL.resolve()} generado con {len(analyzed_photos)} foto(s).")


Listo. /content/analyzed_photos_poc.jsonl generado con 5 foto(s).


## 11. Casos de prueba: criterio, resultado y veredicto

Evidencia de la corrida real mas reciente sobre 5 fotos de galeria
(`Foto1.JPEG` a `Foto5.JPEG`, ver celda de la seccion 8). Cada caso cubre
un escenario distinto de caso feliz o incertidumbre.

| Caso | Criterio esperado | Resultado observado | Veredicto |
|---|---|---|---|
| Foto1 - EXIF completo (iPhone 16, ISO 160, f/1.6) + notas claras | Confianza alta, no marcar revision | confianza 0.99, no marcada | PASS |
| Foto2 - sin datos EXIF (0 tags), con notas claras | No debe fallar; campos EXIF en "-" | proceso completo, EXIF en "-", confianza 0.99, no marcada | PASS |
| Foto3 - notas inconsistentes con la imagen (dice Lima, la imagen muestra Buenos Aires) | Debe detectar la incoherencia y marcar para revision | confianza 0.22, marcada SI, review_reason describe la contradiccion (referencias a "Capital Federal" y "Nicolas Avellaneda" en vez de Miraflores) | PASS |
| Foto4 - sin nota de fotografo (no aparece en notas.txt), sin EXIF | Debe usar "(sin notas)", no fallar | proceso completo, notas "(sin notas)", ambiente inferido como "unknown", confianza 0.98, no marcada | PASS |
| Foto5 - EXIF completo (iPhone 16, ISO 50, f/1.6) + notas claras | Confianza alta, no marcar revision | confianza 0.99, no marcada | PASS |

Nota: en esta corrida Foto5 se resubio con su EXIF original intacto, asi
que dejo de ser un caso "sin EXIF" y paso a ser un segundo caso feliz
(igual que Foto1). La cobertura de "sin EXIF" sigue existiendo gracias a
Foto2, Foto3 y Foto4.

Pendiente: no se probo ningun caso de "fuera de alcance" (Etapa 5,
verificacion de derechos de imagen) porque esa etapa no esta implementada
-- el sistema simplemente no la toca. Tampoco hay un caso de entrada
invalida (carpeta vacia, archivo no soportado).

## 12. Limitaciones y siguiente paso hacia el MVP

**Limitaciones conocidas de este PoC**

- Varias fotos de galeria llegan sin EXIF (apps como WhatsApp limpian esa
  metadata al comprimir/enviar); en esos casos el sistema depende 100% del
  analisis del modelo, sin ningun dato tecnico de camara como respaldo.
- `MODEL_ID = "gpt-5.6-luna"` esta fijo en el codigo y no se valido su
  disponibilidad/estabilidad fuera del entorno de prueba usado.
- No se mide costo por llamada (tokens) ni se limita el gasto por lote.
- El umbral de confianza (`CONFIDENCE_REVIEW_THRESHOLD = 0.80`) esta fijo en
  codigo y no fue calibrado con mas casos reales; con solo 5 fotos no alcanza
  para saber si 0.80 es el corte correcto.
- El esquema de salida no coincide exactamente con el contrato firmado en la
  Sesion 1: `environment` vs `enviroment`, `commercial_classification` vs
  `commercial_clasification`, y `review_reason` es un campo agregado que no
  estaba en el contrato original. Falta reconciliar uno con el otro.
- La Etapa 6 sigue siendo texto en consola, no una interfaz real donde el
  editor pueda aprobar/rechazar.
- El archivo `.jsonl` de salida queda solo en el entorno de ejecucion (por
  ejemplo `/content/` en Colab) y no se versiona como evidencia en el repo.
- Solo se probo con 5 fotos; no valida que el KPI de tiempo por lote se
  sostenga con un lote real de 50+ fotos.

**Siguiente paso hacia el MVP**

1. Reconciliar el esquema de salida con el contrato de la Sesion 1 (o
   actualizar el contrato si los nombres correctos son los del codigo).
2. Correr el mismo pipeline con un lote de tamano real (10-50 fotos) para
   confirmar que el KPI de tiempo se sostiene a escala.
3. Reemplazar el placeholder de consola de la Etapa 6 por una interfaz real
   de aprobacion para el editor/fotografo.
4. Agregar medicion de costo y latencia por foto para poder decidir el
   modelo y el umbral de confianza con datos, no a ojo.
5. Documentar explicitamente el caso "fuera de alcance" (Etapa 5) y agregar
   al menos un caso de prueba de entrada invalida (carpeta vacia, archivo no
   soportado).